# 02 · Congelación del baseline

Este notebook toma el **retrato completo del sistema baseline** — el agente tal y como quedó en la entrega 2 (búsqueda densa, sin middleware, prompt congelado) — y lo deja fijado antes de tocar una sola mejora. Es media tabla del informe y **no se puede reconstruir a posteriori**: cualquier mejora que se cuele antes de esta tirada contamina la comparación para siempre.

Tres tandas, cada una con su papel:

| Tanda | Fichero | Papel |
|---|---|---|
| Propio (20) | `golden/golden_set.jsonl` | **La tabla del informe** (enunciado §5: baseline vs final sobre el golden propio) |
| Oficial (20) | `golden/oficial_20.jsonl` | Diagnóstico y comparabilidad con el material del curso |
| Huecos (5) | `golden/huecos_humo.jsonl` | Ensayo del comportamiento `fuente='ninguna'` (≥2 de las 10 ciegas serán así) |

Coste estimado de la tirada completa: **30-60 céntimos** (45 preguntas a ~0,4-1,2 ¢). Duración: 10-15 minutos, con progreso por pregunta. `temperature=0` en todo: la tirada es reproducible.

**Regla a partir de la última celda:** el baseline queda etiquetado en git (`git tag baseline`) y estos CSV no se regeneran. Toda mejora se mide contra estas cifras.

In [1]:
import json
import sys
from pathlib import Path

RAIZ = Path.cwd() if (Path.cwd() / "agente").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ))

import os
import pandas as pd

if not os.environ.get("OPENROUTER_API_KEY"):
    from getpass import getpass
    os.environ["OPENROUTER_API_KEY"] = getpass("OPENROUTER_API_KEY: ")

from agente import evaluadores, interfaz, retrieval

# El baseline se define por su punto de intercambio: densa. Si esto falla,
# alguien descongeló antes de tiempo.
assert retrieval.buscar_agente is retrieval.buscar_densa, \
    "buscar_agente ya no es la densa: esto NO es el baseline"

propio = evaluadores.cargar_golden(RAIZ / "golden/golden_set.jsonl")
oficial = evaluadores.cargar_golden(RAIZ / "golden/oficial_20.jsonl")
huecos = evaluadores.cargar_golden(RAIZ / "golden/huecos_humo.jsonl")
fams = pd.Series([g["familia"] for g in propio]).value_counts().to_dict()
assert len(propio) == 20 and fams.get("comparativa", 0) >= 6, fams
print(f"propio: {len(propio)} {fams} · oficial: {len(oficial)} · "
      f"huecos: {len(huecos)}")
print("Baseline verificado: search_filings = densa + filtros, sin middleware.")

propio: 20 {'numerica': 7, 'extractiva': 7, 'comparativa': 6} · oficial: 20 · huecos: 5
Baseline verificado: search_filings = densa + filtros, sin middleware.


## Tanda 1 — Golden propio (la tabla del informe)

Cada pregunta corre en su propio `thread_id`; la columna `recall` se mide con el retrieval **del baseline** (densa+filtros sobre la consulta reescrita, que es el régimen en el que el agente consulta). Un error en una pregunta produce una fila con `error`, no un crash.

In [2]:
df_propio = interfaz.evaluar(str(RAIZ / "golden/golden_set.jsonl"),
                             etiqueta="baseline_propio")

  ok  pr-n03      0.38c  cita=None cifra=True tool=True
  ok  pr-n05      0.44c  cita=None cifra=True tool=True
  ok  pr-n07      0.36c  cita=None cifra=True tool=True
  ok  pr-n10      0.38c  cita=None cifra=True tool=True
  ok  pr-n11      0.60c  cita=None cifra=True tool=True
  ok  pr-n12      0.40c  cita=None cifra=True tool=True
  ok  pr-n13      0.58c  cita=None cifra=True tool=True


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  ok  pr-e13      1.25c  cita=True cifra=None tool=True
  ok  pr-e14      0.89c  cita=True cifra=None tool=True
  ok  pr-e15      0.70c  cita=True cifra=None tool=True
  ok  pr-e16      2.14c  cita=True cifra=None tool=True
  ok  pr-e18      1.03c  cita=True cifra=None tool=True
  ok  pr-e19      0.90c  cita=True cifra=None tool=True
  ok  pr-e20      1.28c  cita=True cifra=None tool=True
  ok  pr-c06      1.70c  cita=True cifra=True tool=True
  ok  pr-c09      1.42c  cita=False cifra=False tool=True
  ok  pr-c10      1.11c  cita=False cifra=True tool=True
  ok  pr-c11      5.59c  cita=True cifra=False tool=True
  ok  pr-c12      2.87c  cita=True cifra=True tool=True
  ok  pr-c13      1.55c  cita=True cifra=False tool=True
[baseline_propio] n=20  cita=0.85  cifra=0.77  tool=1.00  recall@5=0.69  coste=1.28¢  latencia=23.5s  tools/pregunta=3.6  errores=0
              cita_ok  cifra_ok  tool_ok
familia                                 
comparativa  0.666667       0.5      1.0
extractiva  

## Tanda 2 — Golden oficial (diagnóstico)

Las mismas 20 con las que el profesor razona en clase. No es la tabla del informe, pero sí el mejor contraste: aquí sabemos qué esperaba el curso de cada pregunta (la trampa de Alphabet en of-012, los huecos de capex en of-018, la guidance de of-017…).

In [8]:
df_oficial = interfaz.evaluar(str(RAIZ / "golden/oficial_20.jsonl"),
                              etiqueta="baseline_oficial")

Deserializing unregistered type agente.agente.RespuestaFinanciera from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('agente.agente', 'RespuestaFinanciera')]


  ok  of-001      2.81c  cita=True cifra=None tool=True
  ok  of-002      1.36c  cita=True cifra=None tool=True
  ok  of-003      1.98c  cita=True cifra=None tool=True
  ok  of-004      1.54c  cita=True cifra=None tool=True
  ok  of-005      1.70c  cita=False cifra=None tool=True
  ok  of-006      1.38c  cita=True cifra=None tool=True
  ok  of-007      0.62c  cita=None cifra=True tool=True
  ok  of-008      0.83c  cita=None cifra=True tool=True
  ok  of-009      0.87c  cita=None cifra=True tool=True
  ok  of-010      0.58c  cita=None cifra=True tool=True
  ok  of-011      0.86c  cita=None cifra=True tool=True
  ok  of-012      0.79c  cita=None cifra=True tool=True
  ok  of-013      0.57c  cita=None cifra=True tool=True
  ok  of-014      7.80c  cita=True cifra=False tool=True
  ok  of-015      1.31c  cita=True cifra=False tool=True
  ok  of-016      1.23c  cita=True cifra=False tool=True
  ok  of-017      1.56c  cita=True cifra=True tool=True
  ok  of-018      2.27c  cita=False cifra=Fa

## Tanda 3 — Huecos (el ensayo de las ciegas)

Cinco preguntas **sin respuesta en el corpus**. Aquí la columna que importa es `cifra_ok` con nuestra mejora de evaluador: `True` solo si el agente respondió sin cifra y con `fuente='ninguna'`; una cifra inventada puntúa `False` (el evaluador del taller devolvía `None` y la invención salía gratis). "Que el agente conteste fuente='ninguna' en vez de inventarse una cifra es la mitad del examen".

In [10]:
df_huecos = interfaz.evaluar(str(RAIZ / "golden/huecos_humo.jsonl"),
                             etiqueta="baseline_huecos")

  ok  hx-01       2.62c  cita=True cifra=True tool=True
  ok  hx-02       4.60c  cita=True cifra=True tool=True
  ok  hx-03       3.39c  cita=None cifra=True tool=True
  ok  hx-04       0.63c  cita=None cifra=True tool=True
  ok  hx-05       0.55c  cita=None cifra=True tool=True
[baseline_huecos] n=5  cita=1.00  cifra=1.00  tool=1.00  recall@5=nan  coste=2.36¢  latencia=5.0s  tools/pregunta=5.4  errores=0
          cita_ok  cifra_ok  tool_ok
familia                             
numerica      1.0       1.0      1.0


## Consolidado y coste real de la tirada

In [11]:
resumen = pd.DataFrame([
    evaluadores.resumir(df_propio, "baseline · propio (20)"),
    evaluadores.resumir(df_oficial, "baseline · oficial (20)"),
    evaluadores.resumir(df_huecos, "baseline · huecos (5)"),
]).set_index("sistema").round(3)
display(resumen)

(RAIZ / "resultados").mkdir(exist_ok=True)
resumen.to_csv(RAIZ / "resultados/baseline_resumen.csv")

coste_total = sum(df["coste_usd"].dropna().sum()
                  for df in (df_propio, df_oficial, df_huecos))
print(f"Coste real de la tirada: {coste_total*100:.1f} ¢")
print("Guardado: resultados/baseline_resumen.csv (+ los tres eval_*.csv)")

print("\nDesglose por familia — propio:")
display(evaluadores.por_familia(df_propio).round(2))
print("Fallos por tanda:")
for nombre, df in [("propio", df_propio), ("oficial", df_oficial),
                   ("huecos", df_huecos)]:
    malas = df[(df[["cita_ok", "cifra_ok", "tool_ok"]] == False).any(axis=1)
               | df["error"].notna()]["id"].tolist()
    print(f"  {nombre}: {malas if malas else 'ninguno'}")

,n,cita_ok,cifra_ok,tool_ok,recall@5,coste_medio_¢,latencia_media_s,tools_por_pregunta,errores
sistema,,,,,,,,,
baseline · propio (20),20,0.846,0.769,1.0,0.692,1.277,23.453,3.60,0
baseline · oficial (20),20,0.769,0.571,1.0,0.769,1.753,17.070,4.75,0
baseline · huecos (5),5,1.000,1.000,1.0,NaN,2.358,4.978,5.40,0


Coste real de la tirada: 72.4 ¢
Guardado: resultados/baseline_resumen.csv (+ los tres eval_*.csv)

Desglose por familia — propio:


,cita_ok,cifra_ok,tool_ok
familia,,,
comparativa,0.67,0.5,1.0
extractiva,1.00,NaN,1.0
numerica,NaN,1.0,1.0


Fallos por tanda:
  propio: ['pr-c09', 'pr-c10', 'pr-c11', 'pr-c13']
  oficial: ['of-005', 'of-014', 'of-015', 'of-016', 'of-018', 'of-019', 'of-020']
  huecos: ninguno


## Inspector de fallos

Para entender un fallo no basta la fila del CSV: hay que **ver la trayectoria**. La celda siguiente define `inspeccionar(id)`: reejecuta esa pregunta en un hilo nuevo con `pretty_trace` (cuesta lo que una pregunta, ~0,5-1 ¢). Clasifica cada fallo en su causa — retrieval (no encontró), enrutado (herramienta equivocada), cita (no ancló) o cifra (número mal) — porque **la causa decide qué mejora lo ataca**: el flip a híbrida ataca retrieval, el middleware ataca cifra, el prompt ataca enrutado. Ese mapa fallo→causa→mejora es la sección central del informe.

In [14]:
TODAS = {g["id"]: g for g in propio + oficial + huecos}

def inspeccionar(ident: str):
    from agente import trazas
    g = TODAS[ident]
    print(f"[{ident}] {g['familia']} · esperada: "
          f"{g.get('cifra_esperada')} · tools: {g['herramienta_esperada']}")
    print(f"P: {g['pregunta']}\n")
    r = interfaz.responder(g["pregunta"], thread_id=f"inspeccion-{ident}")
    trazas.pretty_trace(r)
    print(f"\n[{r['latencia_s']:.1f} s · {r['coste_usd']*100:.2f} ¢]")
    return r

# Ejemplo (descomenta el fallo que quieras mirar):

prueba_1 = inspeccionar("of-012")

[of-012] numerica · esperada: 402836000000.0 · tools: ['get_xbrl_fact']
P: ¿Cuáles fueron los ingresos de Alphabet en 2025?

  1. list_available()
       -> AAPL (Apple Inc.): ejercicios [2024, 2025], items ['1A', '7', '7A', '8'] AMZN (AMAZON COM INC): ejercicios [2024, 2025], items ['1A', '7', '7A', '8'] GOOGL (Alphabet Inc.): ejercicios [2024, 2025], it…
  2. get_xbrl_fact(ticker='GOOGL', fiscal_year=2025, concept='Revenues')
       -> GOOGL FY2025 · Revenues = 402,836,000,000 USD (cierre de ejercicio 2025-12-31, según el 10-K)
  3. RespuestaFinanciera(unidad='USD', fuente='xbrl', concept_xbrl='Revenues', cifra=402836000000, ticker='GOOGL', ejercicio=2025, respuesta='Los ingresos de Alphabet en el ejercicio fiscal 2025 fueron de 402.836.000.000 USD (aproximadamente 402.836 millones de dólares).')
       -> Returning structured response: respuesta='Los ingresos de Alphabet en el ejercicio fiscal 2025 fueron de 402.836.000.000 USD (aproximadamente 402.836 millones de dólares).' cifra=4

## Congelación

Con las tres tandas arriba y los CSV escritos, el baseline se congela **fuera** de este notebook, en el terminal:

```bash
git add .
git commit -m "Baseline congelado: tirada baseline (propio/oficial/huecos) sobre el golden propio final"
git tag -fa baseline -m "Sistema baseline: densa+filtros, sin middleware, prompt v1"
git push && git push --force origin baseline
```

`-f` y `--force` solo son necesarios si el tag `baseline` ya existía de una tirada anterior: mueven el tag al commit nuevo.

A partir del tag: (1) `buscar_agente` pasa a la configuración ganadora del notebook 01, (2) entra el middleware (límites + verificación por concepto), (3) cada mejora se re-mide con `evaluar()` y se compara contra `resultados/eval_baseline_propio.csv`. Nada de esta página se reejecuta después del tag.

In [15]:
print("NOTEBOOK 02 COMPLETADO — baseline retratado.")
print("Siguiente paso en el terminal: commit + git tag baseline + push --tags")
print("Ficheros para compartir: resultados/eval_baseline_propio.csv, "
      "eval_baseline_oficial.csv, eval_baseline_huecos.csv, "
      "baseline_resumen.csv")

NOTEBOOK 02 COMPLETADO — baseline retratado.
Siguiente paso en el terminal: commit + git tag baseline + push --tags
Ficheros para compartir: resultados/eval_baseline_propio.csv, eval_baseline_oficial.csv, eval_baseline_huecos.csv, baseline_resumen.csv
